# CS2227: Artificial Intelligence and Machine Learning
## Lab 5: Support Vector Machines (SVM) with Kernel Tricks

**Objective:** Understand how SVMs find the optimal separating hyperplane and how Kernel Functions allow SVMs to handle non-linear data by projecting it into higher dimensions.

---
### The Core Idea
SVM finds a **hyperplane** that maximizes the **margin** between two classes.

When data is **non-linear** (like concentric circles), a straight line can't separate it. The **Kernel Trick** maps data into a higher-dimensional space where linear separation *is* possible — without explicitly computing the new coordinates.

### Common Kernels:
| Kernel | Formula | Use Case |
|---|---|---|
| Linear | `K(x, x') = xᵀx'` | Linearly separable data |
| Polynomial | `K(x, x') = (γxᵀx' + r)^d` | Curved boundaries |
| RBF (Radial Basis Function) | `K(x, x') = exp(-γ‖x - x'‖²)` | Complex, circular, non-linear data |

> ⚠️ **Important:** SVM uses Euclidean distance (`‖x - x'‖²`), so it is highly sensitive to feature scale. **Always apply `StandardScaler` before training an SVM.**

---
## Step 1: Import Libraries and Define Helper Function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

print("All libraries imported successfully!")

In [ ]:
def plot_decision_boundary(clf, X, y, title):
    """
    Helper function to visualize the decision boundary of an SVM classifier.
    Creates a fine mesh grid, predicts the class for each grid point,
    then draws a filled contour map showing the decision regions.
    """
    plt.figure(figsize=(6, 5))
    ax = plt.gca()

    # Create a mesh grid with step size 0.02 for smooth contour lines
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))

    # Predict for every point in the grid
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot decision regions and data points
    ax.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k', s=25)
    ax.set_title(title)
    ax.set_xticks(())
    ax.set_yticks(())
    plt.show()

print("Helper function defined!")

---
## Step 2: Generate Non-Linear Data (Concentric Circles)

We use `make_circles` to generate data that **cannot** be separated by a straight line — perfect for testing SVM kernels.

- `factor=0.3` controls how close the inner and outer circles are
- `noise=0.1` adds realistic randomness

In [ ]:
# Generate Non-Linear Data (Concentric Circles)
X, y = make_circles(n_samples=300, factor=0.3, noise=0.1, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# --- CRITICAL STEP: Apply Standard Scaling ---
# SVM uses Euclidean distance, so unscaled features will bias results
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train, then transform
X_test_scaled = scaler.transform(X_test)          # Only transform test (no fitting!)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print("Scaling applied successfully!")

---
## Step 3: Linear Kernel SVM

A Linear SVM tries to draw a **straight line** to separate the two classes.

Expected result: **Poor accuracy** — a straight line cannot capture the circular boundary.

In [ ]:
# Train a Linear SVM
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_scaled, y_train)

# Predict and evaluate
linear_preds = svm_linear.predict(X_test_scaled)
linear_acc = accuracy_score(y_test, linear_preds)

print(f"Linear Kernel Accuracy: {linear_acc:.2f}")
print("(Expected to perform poorly on circular data)")

# Visualize the decision boundary
plot_decision_boundary(
    svm_linear, X_train_scaled, y_train,
    f"Linear SVM (Accuracy: {linear_acc:.2f})\nFailed to capture non-linearity"
)

---
## Step 4: Polynomial Kernel SVM

The Polynomial kernel projects data using polynomial combinations of features, allowing **curved decision boundaries**.

- `degree=3` → Cubic polynomial boundary
- `C=1.0` → Regularization parameter (higher C = less tolerance for misclassification)

Expected result: **Better than linear**, but may not perfectly capture the circular shape.

In [ ]:
# Train a Polynomial SVM (degree=3 means cubic curve)
svm_poly = SVC(kernel='poly', degree=3, C=1.0)
svm_poly.fit(X_train_scaled, y_train)

# Predict and evaluate
poly_preds = svm_poly.predict(X_test_scaled)
poly_acc = accuracy_score(y_test, poly_preds)

print(f"Polynomial Kernel Accuracy: {poly_acc:.2f}")

# Visualize the decision boundary
plot_decision_boundary(
    svm_poly, X_train_scaled, y_train,
    f"Polynomial SVM (Accuracy: {poly_acc:.2f})\nModerate non-linear fit"
)

---
## Step 5: RBF (Radial Basis Function) Kernel SVM

The RBF kernel theoretically maps data into **infinite dimensions**, making it extremely flexible.

- `gamma='scale'` → Automatically sets γ based on feature variance (recommended default)
- `C=1.0` → Regularization

Expected result: **Best accuracy** — perfectly captures the concentric circle pattern.

In [ ]:
# Train an RBF SVM
# gamma controls the 'spread' of each point's influence
# C controls regularization (bias-variance tradeoff)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_train_scaled, y_train)

# Predict and evaluate
rbf_preds = svm_rbf.predict(X_test_scaled)
rbf_acc = accuracy_score(y_test, rbf_preds)

print(f"RBF Kernel Accuracy: {rbf_acc:.2f}")

# Visualize the decision boundary
plot_decision_boundary(
    svm_rbf, X_train_scaled, y_train,
    f"RBF Kernel SVM (Accuracy: {rbf_acc:.2f})\nSuccessfully separated using RBF"
)

---
## Step 6: Kernel Comparison Summary

In [ ]:
# Print a summary comparison of all three kernels
print("="*45)
print("       SVM Kernel Comparison Summary")
print("="*45)
print(f"  Linear Kernel Accuracy     : {linear_acc:.2f}")
print(f"  Polynomial Kernel Accuracy : {poly_acc:.2f}")
print(f"  RBF Kernel Accuracy        : {rbf_acc:.2f}")
print("="*45)
print("\nConclusion:")
print("- Linear fails on non-linear data")
print("- Polynomial partially captures curves")
print("- RBF handles the circular boundary best")

---
## Summary

| Kernel | Boundary Shape | Accuracy on Circles | Notes |
|---|---|---|---|
| Linear | Straight line | Low | Can't capture curves |
| Polynomial | Curved (degree 3) | Medium | Better but limited |
| RBF | Infinite-dimensional | High | Best for non-linear data |

**Key Takeaways:**
1. Always use `StandardScaler` before SVM — it's **mandatory** since SVM uses distances.
2. The **RBF kernel** is the go-to for most real-world non-linear problems.
3. The `C` parameter controls regularization: higher C → tighter fit → risk of overfitting.
4. The `gamma` parameter in RBF controls how far the influence of a single training point reaches.